<a href="https://colab.research.google.com/github/nolszewska135/Sztuczna-Inteligencja/blob/main/lab4_21193.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import tensorflow as tf
from sklearn import preprocessing

# ==========================================
# BLOK 1: Wstępne przetwarzanie danych
# ==========================================

# 1. Załadowanie surowych danych z pliku CSV
raw_csv_data = np.loadtxt('/content/Audiobooks_data.csv', delimiter=',')

# Wyodrębniamy wejścia (omijamy ID i Target) oraz targety (ostatnia kolumna)
unscaled_inputs_all = raw_csv_data[:, 1:-1]
targets_all = raw_csv_data[:, -1]

# 2. Przetasowanie danych przed balansowaniem
shuffled_indices = np.arange(unscaled_inputs_all.shape[0])
np.random.shuffle(shuffled_indices)

unscaled_inputs_all = unscaled_inputs_all[shuffled_indices]
targets_all = targets_all[shuffled_indices]

# 3. Balansowanie zbioru danych (Equal Priors - tyle samo 0 i 1)
num_one_targets = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove = []

for i in range(targets_all.shape[0]):
    if targets_all[i] == 0:
        zero_targets_counter += 1
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i)

unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors = np.delete(targets_all, indices_to_remove, axis=0)

# 4. Standaryzacja/Skalowanie danych (kluczowe dla sieci neuronowych)
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

# 5. Ponowne przetasowanie znormalizowanych danych
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equal_priors[shuffled_indices]

# 6. Podział na zbiory: Treningowy (80%), Walidacyjny (10%), Testowy (10%)
samples_count = shuffled_inputs.shape[0]

train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)
test_samples_count = samples_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count:]

# Kontrolne wyświetlenie zbalansowania podzbiorów
print("--- Sprawdzenie zbalansowania danych ---")
print(f"Trening: jedynki = {np.sum(train_targets)}, razem = {train_samples_count}, stosunek = {np.sum(train_targets) / train_samples_count:.4f}")
print(f"Walidacja: jedynki = {np.sum(validation_targets)}, razem = {validation_samples_count}, stosunek = {np.sum(validation_targets) / validation_samples_count:.4f}")
print(f"Test: jedynki = {np.sum(test_targets)}, razem = {test_samples_count}, stosunek = {np.sum(test_targets) / test_samples_count:.4f}\n")

# Zapis do plików .npz (zgodnie z instrukcją laboratorium)
np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)


# ==========================================
# BLOK 2: Budowa, trening i ewaluacja modelu
# ==========================================

# Załadowanie danych z przygotowanych plików i konwersja typów
npz_train = np.load('Audiobooks_data_train.npz')
train_inputs = npz_train['inputs'].astype(np.float32)
train_targets = npz_train['targets'].astype(np.int32)

npz_val = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz_val['inputs'].astype(np.float32)
validation_targets = npz_val['targets'].astype(np.int32)

npz_test = np.load('Audiobooks_data_test.npz')
test_inputs = npz_test['inputs'].astype(np.float32)
test_targets = npz_test['targets'].astype(np.int32)

# Konfiguracja ulepszonej architektury modelu
input_size = 10
output_size = 2
hidden_layer_size = 128  # Zwiększono rozmiar warstwy z 50 do 128

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu', input_shape=(input_size,)),
    tf.keras.layers.Dropout(0.2),  # Dodano Dropout zapobiegający przeuczeniu
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dropout(0.2),  # Dodano Dropout zapobiegający przeuczeniu
    tf.keras.layers.Dense(output_size, activation='softmax')
])

# Kompilacja modelu z jawnym określeniem learning rate optimizera Adam
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Konfiguracja treningu
batch_size = 64  # Zmniejszono z 100 na 64 dla dokładniejszego dostrajania wag
max_epochs = 100

# Zwiększono cierpliwość (patience) do 10 epok, aby dać modelowi szansę na lepszą zbieżność
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("--- Rozpoczęcie uczenia modelu ---")
model.fit(
    train_inputs,
    train_targets,
    batch_size=batch_size,
    epochs=max_epochs,
    callbacks=[early_stopping],
    validation_data=(validation_inputs, validation_targets),
    verbose=1
)

# Ewaluacja na zbiorze testowym
print("\n--- Wynik końcowy na zbiorze testowym ---")
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets, verbose=0)
print('Test loss: {:.4f}. Test accuracy: {:.2f}%'.format(test_loss, test_accuracy * 100.))


--- Sprawdzenie zbalansowania danych ---
Trening: jedynki = 1796.0, razem = 3579, stosunek = 0.5018
Walidacja: jedynki = 224.0, razem = 447, stosunek = 0.5011
Test: jedynki = 217.0, razem = 448, stosunek = 0.4844

--- Rozpoczęcie uczenia modelu ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6943 - loss: 0.5584 - val_accuracy: 0.7539 - val_loss: 0.4641
Epoch 2/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7597 - loss: 0.4497 - val_accuracy: 0.7651 - val_loss: 0.4295
Epoch 3/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7731 - loss: 0.4171 - val_accuracy: 0.7897 - val_loss: 0.4119
Epoch 4/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7815 - loss: 0.4057 - val_accuracy: 0.7897 - val_loss: 0.3969
Epoch 5/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7793 - loss: 0.4128 - val_accuracy: 0.7852 - val_loss: 0.4013
Epoch 6/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7904 - loss: 0.3946 - val_accuracy: 0.7919 - val_loss: 0.3972
Epoch 7/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7865 - loss: 0.3898 - val_accuracy: 0.8031 - val_loss: 0.4024
Epoch 8/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7902 - loss: 0.3947 - val_accuracy: 0.7919 - v

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
